# Mini experimento de prompts para code smells

Este notebook prepara uma **amostra piloto** do dataset para ajudar a escolher os melhores prompts do experimento principal.

A ideia aqui é:

1. ler o arquivo `MLCQ_ground_truth_completo.xlsx`
2. transformar a base em **1 linha por trecho**
3. separar os trechos por tipo:
   - `class` → smells: `blob` e `data class`
   - `function` → smells: `long method` e `feature envy`
4. sortear **10 trechos com smell** e **10 trechos sem smell** para cada smell
5. opcionalmente baixar o trecho de código Java a partir do link do GitHub
6. salvar amostras em CSV para usar no mini experimento de prompts

> Observação importante: o arquivo original tem várias linhas para o mesmo trecho, porque um mesmo `sample_id` pode aparecer em mais de um smell e/ou ter sido revisado por mais de um desenvolvedor. Por isso, o primeiro passo é consolidar a base.


In [ ]:
# Se precisar instalar dependências, rode esta célula:

# python3 -m venv .venv
# source .venv/bin/activate

# !pip install pandas openpyxl requests

In [3]:
import hashlib
import json
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [4]:
# Ajuste o caminho se necessário
ARQUIVO_XLSX = "MLCQ_ground_truth_completo.xlsx"

df_raw = pd.read_excel(ARQUIVO_XLSX)
print(df_raw.shape)
display(df_raw.head())

(8711, 15)


,sample_id,type,smell,code_name,repository,commit_hash,path,start_line,end_line,link,is_from_industry_relevant_project,total_avaliacoes,votos_tem_smell,votos_nao_tem_smell,veredito_final
0,3698323,class,blob,com.amazon.ask.dispatcher.request.handler.impl.PlaybackNearlyFinishedRequestHandler,git@github.com:alexa/alexa-skills-kit-sdk-for-java.git,bf1e9ccc50d1f3f8408f887f70197ee288fd4bd9,/ask-sdk-core/src/com/amazon/ask/dispatcher/request/handler/impl/PlaybackNearlyFinishedRequestHandler.java,26,59,https://github.com/alexa/alexa-skills-kit-sdk-for-java/blob/bf1e9ccc50d1f3f8408f887f70197ee288fd4bd9/ask-sdk-core/sr...,1.0,1,0,1,0.0
1,3698323,class,data class,com.amazon.ask.dispatcher.request.handler.impl.PlaybackNearlyFinishedRequestHandler,git@github.com:alexa/alexa-skills-kit-sdk-for-java.git,bf1e9ccc50d1f3f8408f887f70197ee288fd4bd9,/ask-sdk-core/src/com/amazon/ask/dispatcher/request/handler/impl/PlaybackNearlyFinishedRequestHandler.java,26,59,https://github.com/alexa/alexa-skills-kit-sdk-for-java/blob/bf1e9ccc50d1f3f8408f887f70197ee288fd4bd9/ask-sdk-core/sr...,1.0,1,0,1,0.0
2,3698602,function,feature envy,com.amazon.ask.request.mapper.impl.BaseRequestMapper.Builder#build,git@github.com:alexa/alexa-skills-kit-sdk-for-java.git,bf1e9ccc50d1f3f8408f887f70197ee288fd4bd9,/ask-sdk-runtime/src/com/amazon/ask/request/mapper/impl/BaseRequestMapper.java,79,81,https://github.com/alexa/alexa-skills-kit-sdk-for-java/blob/bf1e9ccc50d1f3f8408f887f70197ee288fd4bd9/ask-sdk-runtime...,1.0,1,0,1,0.0
3,3698602,function,long method,com.amazon.ask.request.mapper.impl.BaseRequestMapper.Builder#build,git@github.com:alexa/alexa-skills-kit-sdk-for-java.git,bf1e9ccc50d1f3f8408f887f70197ee288fd4bd9,/ask-sdk-runtime/src/com/amazon/ask/request/mapper/impl/BaseRequestMapper.java,79,81,https://github.com/alexa/alexa-skills-kit-sdk-for-java/blob/bf1e9ccc50d1f3f8408f887f70197ee288fd4bd9/ask-sdk-runtime...,1.0,1,0,1,0.0
4,3698665,function,feature envy,"com.amazon.ask.builder.impl.AbstractSkillBuilder#addExceptionHandler GenericExceptionHandler<Input, Output>",git@github.com:alexa/alexa-skills-kit-sdk-for-java.git,bf1e9ccc50d1f3f8408f887f70197ee288fd4bd9,/ask-sdk-runtime/src/com/amazon/ask/builder/impl/AbstractSkillBuilder.java,91,94,https://github.com/alexa/alexa-skills-kit-sdk-for-java/blob/bf1e9ccc50d1f3f8408f887f70197ee288fd4bd9/ask-sdk-runtime...,1.0,1,0,1,0.0


In [5]:
df_raw.columns.tolist()

['sample_id',
 'type',
 'smell',
 'code_name',
 'repository',
 'commit_hash',
 'path',
 'start_line',
 'end_line',
 'link',
 'is_from_industry_relevant_project',
 'total_avaliacoes',
 'votos_tem_smell',
 'votos_nao_tem_smell',
 'veredito_final']

## 1. Tratamento de duplicados

Vamos transformar a base em **1 linha por trecho** (`sample_id`), com colunas binárias para cada smell relevante.


In [6]:
CLASS_SMELLS = ["blob", "data class"]
FUNCTION_SMELLS = ["long method", "feature envy"]

def normalize_smell_name(smell: str) -> str:
    return str(smell).strip().lower().replace("_", " ")

df = df_raw.copy()
df["smell"] = df["smell"].map(normalize_smell_name)
df["type"] = df["type"].astype(str).str.strip().str.lower()

index_cols = [
    "sample_id",
    "type",
    "code_name",
    "repository",
    "commit_hash",
    "path",
    "start_line",
    "end_line",
    "link",
    "is_from_industry_relevant_project",
]

base = df[index_cols].drop_duplicates().copy()

pivot = (
    df.pivot_table(
        index="sample_id",
        columns="smell",
        values="veredito_final",
        aggfunc="max",
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

prepared = base.merge(pivot, on="sample_id", how="left")

for smell in CLASS_SMELLS + FUNCTION_SMELLS:
    if smell not in prepared.columns:
        prepared[smell] = pd.NA

for smell in CLASS_SMELLS:
    prepared.loc[prepared["type"] == "function", smell] = pd.NA
for smell in FUNCTION_SMELLS:
    prepared.loc[prepared["type"] == "class", smell] = pd.NA

for smell in CLASS_SMELLS:
    prepared.loc[(prepared["type"] == "class") & (prepared[smell].isna()), smell] = 0
for smell in FUNCTION_SMELLS:
    prepared.loc[(prepared["type"] == "function") & (prepared[smell].isna()), smell] = 0

prepared["start_line"] = prepared["start_line"].astype(int)
prepared["end_line"] = prepared["end_line"].astype(int)

prepared = prepared.sort_values(["type", "sample_id"]).reset_index(drop=True)

print("Linhas consolidadas:", len(prepared))
display(prepared.head())

Linhas consolidadas: 4364


,sample_id,type,code_name,repository,commit_hash,path,start_line,end_line,link,is_from_industry_relevant_project,blob,data class,feature envy,long method
0,3698323,class,com.amazon.ask.dispatcher.request.handler.impl.PlaybackNearlyFinishedRequestHandler,git@github.com:alexa/alexa-skills-kit-sdk-for-java.git,bf1e9ccc50d1f3f8408f887f70197ee288fd4bd9,/ask-sdk-core/src/com/amazon/ask/dispatcher/request/handler/impl/PlaybackNearlyFinishedRequestHandler.java,26,59,https://github.com/alexa/alexa-skills-kit-sdk-for-java/blob/bf1e9ccc50d1f3f8408f887f70197ee288fd4bd9/ask-sdk-core/sr...,1.0,0.0,0.0,NaN,NaN
1,3699849,class,com.alibaba.otter.canal.spi.CanalMetricsService,git@github.com:alibaba/canal.git,08167c95c767fd3c9879584c0230820a8476a7a7,/server/src/main/java/com/alibaba/otter/canal/spi/CanalMetricsService.java,13,47,https://github.com/alibaba/canal/blob/08167c95c767fd3c9879584c0230820a8476a7a7/server/src/main/java/com/alibaba/otte...,1.0,0.0,0.0,NaN,NaN
2,3700666,class,com.alibaba.otter.canal.protocol.CanalEntry.RowChange.Builder,git@github.com:alibaba/canal.git,08167c95c767fd3c9879584c0230820a8476a7a7,/protocol/src/main/java/com/alibaba/otter/canal/protocol/CanalEntry.java,8477,9689,https://github.com/alibaba/canal/blob/08167c95c767fd3c9879584c0230820a8476a7a7/protocol/src/main/java/com/alibaba/ot...,1.0,1.0,1.0,NaN,NaN
3,3702984,class,com.alibaba.otter.canal.client.adapter.hbase.support.PhTypeUtil,git@github.com:alibaba/canal.git,08167c95c767fd3c9879584c0230820a8476a7a7,/client-adapter/hbase/src/main/java/com/alibaba/otter/canal/client/adapter/hbase/support/PhTypeUtil.java,21,609,https://github.com/alibaba/canal/blob/08167c95c767fd3c9879584c0230820a8476a7a7/client-adapter/hbase/src/main/java/co...,1.0,1.0,0.0,NaN,NaN
4,3705164,class,com.alibaba.otter.canal.protocol.ClientIdentity,git@github.com:alibaba/canal.git,08167c95c767fd3c9879584c0230820a8476a7a7,/protocol/src/main/java/com/alibaba/otter/canal/protocol/ClientIdentity.java,14,105,https://github.com/alibaba/canal/blob/08167c95c767fd3c9879584c0230820a8476a7a7/protocol/src/main/java/com/alibaba/ot...,1.0,0.0,0.0,NaN,NaN


In [7]:
print("Trechos únicos por tipo:")
display(prepared["type"].value_counts(dropna=False).to_frame("qtd"))

print("\nDistribuição dos smells em classes:")
display(prepared.loc[prepared["type"] == "class", CLASS_SMELLS].fillna(0).astype(int).sum().to_frame("positivos"))

print("\nDistribuição dos smells em funções:")
display(prepared.loc[prepared["type"] == "function", FUNCTION_SMELLS].fillna(0).astype(int).sum().to_frame("positivos"))

Trechos únicos por tipo:


,qtd
type,
function,2224
class,2140



Distribuição dos smells em classes:


,positivos
blob,226
data class,279



Distribuição dos smells em funções:


,positivos
long method,239
feature envy,63


## 2. Funções auxiliares para amostragem

A função abaixo sorteia, para um smell específico, uma amostra com:

- **10 trechos com smell**
- **10 trechos sem smell**

Você pode alterar `n_each` se quiser outro tamanho.


In [8]:
def sample_for_smell(
    data: pd.DataFrame,
    snippet_type: str,
    smell_col: str,
    n_each: int = 10,
    seed: int = 42,
) -> pd.DataFrame:
    subset = data[data["type"] == snippet_type].copy()
    subset[smell_col] = subset[smell_col].fillna(0).astype(int)

    positive = subset[subset[smell_col] == 1].copy()
    negative = subset[subset[smell_col] == 0].copy()

    n_pos = min(n_each, len(positive))
    n_neg = min(n_each, len(negative))

    sample_pos = positive.sample(n=n_pos, random_state=seed) if n_pos > 0 else positive.head(0)
    sample_neg = negative.sample(n=n_neg, random_state=seed) if n_neg > 0 else negative.head(0)

    out = pd.concat([sample_pos, sample_neg], ignore_index=True)
    out["target_smell"] = smell_col
    out["target_label"] = out[smell_col].astype(int)
    out["group_label"] = out["target_label"].map({1: "com_smell", 0: "sem_smell"})

    cols_front = [
        "sample_id",
        "type",
        "target_smell",
        "target_label",
        "group_label",
        "code_name",
        "repository",
        "commit_hash",
        "path",
        "start_line",
        "end_line",
        "link",
        "is_from_industry_relevant_project",
    ]
    other_cols = [c for c in out.columns if c not in cols_front]
    return out[cols_front + other_cols].sort_values(["target_label", "sample_id"], ascending=[False, True]).reset_index(drop=True)

## 3. Criando as amostras do mini experimento

Aqui vamos gerar 4 amostras:

- `blob`
- `data class`
- `long method`
- `feature envy`

Cada uma com 10 exemplos positivos e 10 negativos, quando houver quantidade suficiente.


In [9]:
amostra_blob = sample_for_smell(prepared, snippet_type="class", smell_col="blob", n_each=10, seed=42)
amostra_data_class = sample_for_smell(prepared, snippet_type="class", smell_col="data class", n_each=10, seed=42)
amostra_long_method = sample_for_smell(prepared, snippet_type="function", smell_col="long method", n_each=10, seed=42)
amostra_feature_envy = sample_for_smell(prepared, snippet_type="function", smell_col="feature envy", n_each=10, seed=42)

print("Blob:", amostra_blob.shape)
print("Data Class:", amostra_data_class.shape)
print("Long Method:", amostra_long_method.shape)
print("Feature Envy:", amostra_feature_envy.shape)

Blob: (20, 17)
Data Class: (20, 17)
Long Method: (20, 17)
Feature Envy: (20, 17)


In [10]:
display(amostra_blob.head(5))
display(amostra_data_class.head(5))
display(amostra_long_method.head(5))
display(amostra_feature_envy.head(5))

,sample_id,type,target_smell,target_label,group_label,code_name,repository,commit_hash,path,start_line,end_line,link,is_from_industry_relevant_project,blob,data class,feature envy,long method
0,3908492,class,blob,1,com_smell,org.apache.archiva.consumers.metadata.ArchivaMetadataCreationConsumer,git@github.com:apache/archiva.git,d1242030bf232c0d9b68e4402188ee261924bf4b,/archiva-modules/archiva-base/archiva-consumers/archiva-metadata-consumer/src/main/java/org/apache/archiva/consumers...,59,264,https://github.com/apache/archiva/blob/d1242030bf232c0d9b68e4402188ee261924bf4b/archiva-modules/archiva-base/archiva...,1.0,1,0.0,NaN,NaN
1,5740146,class,blob,1,com_smell,org.apache.storm.generated.Assignment,git@github.com:apache/storm.git,dc56e32f3dcdd9396a827a85029d60ed97474786,/storm-client/src/jvm/org/apache/storm/generated/Assignment.java,26,1404,https://github.com/apache/storm/blob/dc56e32f3dcdd9396a827a85029d60ed97474786/storm-client/src/jvm/org/apache/storm/...,1.0,1,1.0,NaN,NaN
2,6772589,class,blob,1,com_smell,com.facebook.ads.sdk.Page.APIRequestCreateMessengerProfile,git@github.com:facebook/facebook-java-business-sdk.git,561f1a75e1220b55a160a1b92b0187f72be9cd08,/src/main/java/com/facebook/ads/sdk/Page.java,25416,25594,https://github.com/facebook/facebook-java-business-sdk/blob/561f1a75e1220b55a160a1b92b0187f72be9cd08/src/main/java/c...,1.0,1,0.0,NaN,NaN
3,7356691,class,blob,1,com_smell,com.microsoft.tfs.client.clc.vc.commands.CommandShelvesets,git@github.com:Microsoft/team-explorer-everywhere.git,89ab2a4847aec8ec2afdf36c3f6287dd03bd558d,/source/com.microsoft.tfs.client.clc/src/com/microsoft/tfs/client/clc/vc/commands/CommandShelvesets.java,34,171,https://github.com/Microsoft/team-explorer-everywhere/blob/89ab2a4847aec8ec2afdf36c3f6287dd03bd558d/source/com.micro...,1.0,1,0.0,NaN,NaN
4,8119000,class,blob,1,com_smell,org.eclipse.xtext.util.Strings,git@github.com:eclipse/xtext-core.git,e04964e4c2a3e0338c0079bd8333688835e77c31,/org.eclipse.xtext.util/src/org/eclipse/xtext/util/Strings.java,23,475,https://github.com/eclipse/xtext-core/blob/e04964e4c2a3e0338c0079bd8333688835e77c31/org.eclipse.xtext.util/src/org/e...,1.0,1,0.0,NaN,NaN


,sample_id,type,target_smell,target_label,group_label,code_name,repository,commit_hash,path,start_line,end_line,link,is_from_industry_relevant_project,blob,data class,feature envy,long method
0,4109372,class,data class,1,com_smell,org.apache.brooklyn.core.sensor.BasicAttributeSensor,git@github.com:apache/brooklyn-server.git,880eb1da00f6358d7fd76d065322e3685bfb1a04,/core/src/main/java/org/apache/brooklyn/core/sensor/BasicAttributeSensor.java,31,67,https://github.com/apache/brooklyn-server/blob/880eb1da00f6358d7fd76d065322e3685bfb1a04/core/src/main/java/org/apach...,1.0,0.0,1,NaN,NaN
1,5775703,class,data class,1,com_smell,org.apache.syncope.common.lib.to.PagedResult,git@github.com:apache/syncope.git,114c412afbfba24ffb4fbc804e5308a823a16a78,/common/idrepo/lib/src/main/java/org/apache/syncope/common/lib/to/PagedResult.java,35,135,https://github.com/apache/syncope/blob/114c412afbfba24ffb4fbc804e5308a823a16a78/common/idrepo/lib/src/main/java/org/...,1.0,0.0,1,NaN,NaN
2,6293765,class,data class,1,com_smell,org.eclipse.jetty.websocket.api.UpgradeRequest,git@github.com:eclipse/jetty.project.git,65528f76c5ef6ddca11385f9721c8f0bc5f2eed7,/jetty-websocket/websocket-api/src/main/java/org/eclipse/jetty/websocket/api/UpgradeRequest.java,32,323,https://github.com/eclipse/jetty.project/blob/65528f76c5ef6ddca11385f9721c8f0bc5f2eed7/jetty-websocket/websocket-api...,1.0,0.0,1,NaN,NaN
3,6879905,class,data class,1,com_smell,com.sun.tools.javac.tree.DCTree.DCSerialField,git@github.com:google/error-prone-javac.git,a53d069bbdb2c60232ed3811c19b65e41c3e60e0,/src/jdk.compiler/share/classes/com/sun/tools/javac/tree/DCTree.java,732,767,https://github.com/google/error-prone-javac/blob/a53d069bbdb2c60232ed3811c19b65e41c3e60e0/src/jdk.compiler/share/cla...,0.0,0.0,1,NaN,NaN
4,6923657,class,data class,1,com_smell,com.google.googlejavaformat.intellij.GoogleJavaFormatSettings.State,git@github.com:google/google-java-format.git,df76e0c7fe82711c8600768fca19d0ebaf2ca3d2,/idea_plugin/src/com/google/googlejavaformat/intellij/GoogleJavaFormatSettings.java,79,105,https://github.com/google/google-java-format/blob/df76e0c7fe82711c8600768fca19d0ebaf2ca3d2/idea_plugin/src/com/googl...,0.5,0.0,1,NaN,NaN


,sample_id,type,target_smell,target_label,group_label,code_name,repository,commit_hash,path,start_line,end_line,link,is_from_industry_relevant_project,blob,data class,feature envy,long method
0,3922349,function,long method,1,com_smell,"org.apache.aries.rsa.discovery.endpoint.PropertiesMapper#fromProps Map<String, Object>",git@github.com:apache/aries-rsa.git,f5aa5ca62c3948d7e471c3a839089180650cf4f2,/discovery/local/src/main/java/org/apache/aries/rsa/discovery/endpoint/PropertiesMapper.java,233,280,https://github.com/apache/aries-rsa/blob/f5aa5ca62c3948d7e471c3a839089180650cf4f2/discovery/local/src/main/java/org/...,1.0,NaN,NaN,0.0,1
1,3986717,function,long method,1,com_smell,org.apache.polygene.api.util.Classes.simpleGenericNameOf StringBuilder|Type,git@github.com:apache/attic-polygene-java.git,031beef870302a0bd01bd5895ce849e00f2d5d5b,/core/api/src/main/java/org/apache/polygene/api/util/Classes.java,288,342,https://github.com/apache/attic-polygene-java/blob/031beef870302a0bd01bd5895ce849e00f2d5d5b/core/api/src/main/java/o...,0.0,NaN,NaN,0.0,1
2,4426200,function,long method,1,com_smell,org.apache.eagle.jpm.spark.history.status.JobHistoryZKStateManager#resetApplications,git@github.com:apache/eagle.git,7ac9421c2c27d12ae88f001866b4444310fcaa3f,/eagle-jpm/eagle-jpm-spark-history/src/main/java/org/apache/eagle/jpm/spark/history/status/JobHistoryZKStateManager....,103,133,https://github.com/apache/eagle/blob/7ac9421c2c27d12ae88f001866b4444310fcaa3f/eagle-jpm/eagle-jpm-spark-history/src/...,0.5,NaN,NaN,1.0,1
3,5717998,function,long method,1,com_smell,org.apache.sysml.utils.Explain.countCompiledInstructions ProgramBlock|ExplainCounts|boolean|boolean|boolean,git@github.com:apache/systemml.git,7fba4b29d653747a9ed038d282954a44fea3031c,/src/main/java/org/apache/sysml/utils/Explain.java,1103,1141,https://github.com/apache/systemml/blob/7fba4b29d653747a9ed038d282954a44fea3031c/src/main/java/org/apache/sysml/util...,1.0,NaN,NaN,0.0,1
4,5950764,function,long method,1,com_smell,com.amazonaws.kinesisvideo.parser.examples.lambda.KinesisVideoRekognitionLambdaExample#processSingleRecord Record,git@github.com:aws/amazon-kinesis-video-streams-parser-library.git,c54e6388f0f8e2769369995b9710ffd03cf9cc67,/src/main/java/com/amazonaws/kinesisvideo/parser/examples/lambda/KinesisVideoRekognitionLambdaExample.java,236,295,https://github.com/aws/amazon-kinesis-video-streams-parser-library/blob/c54e6388f0f8e2769369995b9710ffd03cf9cc67/src...,0.5,NaN,NaN,1.0,1


,sample_id,type,target_smell,target_label,group_label,code_name,repository,commit_hash,path,start_line,end_line,link,is_from_industry_relevant_project,blob,data class,feature envy,long method
0,3807471,function,feature envy,1,com_smell,"org.apache.activemq.artemis.core.management.impl.AddressControlImpl#sendMessage Map<String, String>|int|String|boole...",git@github.com:apache/activemq-artemis.git,5bd5c610195d6f4a3dd1ac28170727003f8a5a54,/artemis-server/src/main/java/org/apache/activemq/artemis/core/management/impl/AddressControlImpl.java,347,363,https://github.com/apache/activemq-artemis/blob/5bd5c610195d6f4a3dd1ac28170727003f8a5a54/artemis-server/src/main/jav...,1.0,NaN,NaN,1,0.0
1,3951549,function,feature envy,1,com_smell,org.apache.apex.malhar.kudu.AbstractKuduOutputOperator#performCommonProcessing Operation|KuduExecutionContext,git@github.com:apache/apex-malhar.git,1acaf15f425d72f19bb590c667987ed5d81d7f25,/kudu/src/main/java/org/apache/apex/malhar/kudu/AbstractKuduOutputOperator.java,234,347,https://github.com/apache/apex-malhar/blob/1acaf15f425d72f19bb590c667987ed5d81d7f25/kudu/src/main/java/org/apache/ap...,0.5,NaN,NaN,1,1.0
2,4514232,function,feature envy,1,com_smell,org.apache.felix.framework.SecurityActivator#start BundleContext,git@github.com:apache/felix.git,a132994b250751d4ba3b115ee070ba397d9840ca,/framework.security/src/main/java/org/apache/felix/framework/SecurityActivator.java,99,220,https://github.com/apache/felix/blob/a132994b250751d4ba3b115ee070ba397d9840ca/framework.security/src/main/java/org/a...,1.0,NaN,NaN,1,1.0
3,4784081,function,feature envy,1,com_smell,org.apache.sentry.hdfs.service.thrift.SentryHDFSService.handle_hms_notification_result.handle_hms_notification_resul...,git@github.com:apache/incubator-sentry.git,4643f988a5e0ce2b9749e6365edea3a16482de86,/sentry-hdfs/sentry-hdfs-common/src/gen/thrift/gen-javabean/org/apache/sentry/hdfs/service/thrift/SentryHDFSService....,1004,1010,https://github.com/apache/incubator-sentry/blob/4643f988a5e0ce2b9749e6365edea3a16482de86/sentry-hdfs/sentry-hdfs-com...,0.0,NaN,NaN,1,0.0
4,5340257,function,feature envy,1,com_smell,org.apache.nifi.controller.repository.VolatileContentRepository#exportTo ContentClaim|Path|boolean|long|long,git@github.com:apache/nifi.git,c8eff590efa3babcda0b755009224dcac168708b,/nifi-nar-bundles/nifi-framework-bundle/nifi-framework/nifi-framework-core/src/main/java/org/apache/nifi/controller/...,397,418,https://github.com/apache/nifi/blob/c8eff590efa3babcda0b755009224dcac168708b/nifi-nar-bundles/nifi-framework-bundle/...,1.0,NaN,NaN,1,0.0


## 4. Opcional: baixar automaticamente o trecho Java pelo link do GitHub

Se você quiser já levar para o mini experimento os **trechos de código recortados** entre `start_line` e `end_line`, use as células abaixo.

Isso funciona quando o campo `link` aponta para um arquivo do GitHub no formato `github.com/.../blob/...`.


In [11]:
def raw_github_url(link: str) -> str:
    if "github.com" not in link:
        raise ValueError(f"Link não suportado: {link}")

    cleaned = link.split("#")[0]
    parsed = urlparse(cleaned)
    parts = parsed.path.strip("/").split("/")

    if len(parts) < 5 or parts[2] != "blob":
        raise ValueError(f"Formato de link inesperado: {link}")

    owner, repo, _, commit_hash, *file_parts = parts
    return f"https://raw.githubusercontent.com/{owner}/{repo}/{commit_hash}/{'/'.join(file_parts)}"


def fetch_code_snippet(row: pd.Series, cache_dir: str | Path = "code_cache") -> str:
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    digest = hashlib.md5(str(row["link"]).encode("utf-8")).hexdigest()
    cache_file = cache_dir / f"{digest}.java"

    if cache_file.exists():
        full_text = cache_file.read_text(encoding="utf-8")
    else:
        raw_url = raw_github_url(str(row["link"]))
        response = requests.get(raw_url, timeout=30)
        response.raise_for_status()
        full_text = response.text
        cache_file.write_text(full_text, encoding="utf-8")

    lines = full_text.splitlines()
    start = max(int(row["start_line"]) - 1, 0)
    end = min(int(row["end_line"]), len(lines))
    return "\n".join(lines[start:end])


def attach_code(df_sample: pd.DataFrame, cache_dir: str | Path = "code_cache") -> pd.DataFrame:
    snippets = []
    errors = []

    for _, row in df_sample.iterrows():
        try:
            snippets.append(fetch_code_snippet(row, cache_dir=cache_dir))
            errors.append(None)
        except Exception as exc:
            snippets.append("")
            errors.append(str(exc))

    out = df_sample.copy()
    out["code"] = snippets
    out["download_error"] = errors
    return out

In [12]:
# Rode esta célula se quiser anexar os trechos de código às amostras.
# Dependendo da conexão, pode demorar um pouco.

amostra_blob_code = attach_code(amostra_blob)
amostra_data_class_code = attach_code(amostra_data_class)
amostra_long_method_code = attach_code(amostra_long_method)
amostra_feature_envy_code = attach_code(amostra_feature_envy)

display(amostra_blob_code[["sample_id", "target_smell", "target_label", "code", "download_error"]].head(3))

,sample_id,target_smell,target_label,code,download_error
0,3908492,blob,1,"@Service (""knownRepositoryContentConsumer#create-archiva-metadata"")\n@Scope (""prototype"")\npublic class ArchivaMetad...",None
1,5740146,blob,1,"@SuppressWarnings({""cast"", ""rawtypes"", ""serial"", ""unchecked"", ""unused""})\n@javax.annotation.Generated(value = ""Autog...",None
2,6772589,blob,1,public static class APIRequestCreateMessengerProfile extends APIRequest<Page> {\n\n Page lastResponse = null;\n...,None


## 5. Unificando tudo em uma base única do piloto (OPCIONAL)

Aqui juntamos todas as amostras numa única tabela.  
Você pode escolher:

- usar as tabelas **sem código anexado**
- ou usar as tabelas **com a coluna `code`**


In [ ]:
# Escolha uma das opções abaixo.

# OPÇÃO 1: sem coluna de código
piloto_sem_codigo = pd.concat(
    [amostra_blob, amostra_data_class, amostra_long_method, amostra_feature_envy],
    ignore_index=True,
)

# OPÇÃO 2: com coluna de código
piloto_com_codigo = pd.concat(
    [amostra_blob_code, amostra_data_class_code, amostra_long_method_code, amostra_feature_envy_code],
    ignore_index=True,
)

print("Piloto sem código:", piloto_sem_codigo.shape)
print("Piloto com código:", piloto_com_codigo.shape)

In [ ]:
display(
    piloto_com_codigo[
        ["sample_id", "type", "target_smell", "target_label", "group_label", "code_name", "download_error"]
    ].head(10)
)

## 6. Salvando as amostras em CSV (OPCIONAL)

Esses arquivos podem ser usados no seu mini experimento para testar diferentes prompts.


In [ ]:
OUTPUT_DIR = Path("amostras_piloto")
OUTPUT_DIR.mkdir(exist_ok=True)

amostra_blob.to_csv(OUTPUT_DIR / "amostra_blob.csv", index=False)
amostra_data_class.to_csv(OUTPUT_DIR / "amostra_data_class.csv", index=False)
amostra_long_method.to_csv(OUTPUT_DIR / "amostra_long_method.csv", index=False)
amostra_feature_envy.to_csv(OUTPUT_DIR / "amostra_feature_envy.csv", index=False)

piloto_sem_codigo.to_csv(OUTPUT_DIR / "piloto_unificado_sem_codigo.csv", index=False)
piloto_com_codigo.to_csv(OUTPUT_DIR / "piloto_unificado_com_codigo.csv", index=False)

print("Arquivos salvos em:", OUTPUT_DIR.resolve())
list(OUTPUT_DIR.iterdir())

## 7. Próximo passo sugerido

Depois de gerar `piloto_unificado_com_codigo.csv`, o próximo passo é criar uma tabela com algo assim:

- `sample_id`
- `type`
- `target_smell`
- `target_label`
- `prompt_version`
- `model_name`
- `model_output`

Aí você consegue comparar qual prompt teve melhor desempenho para cada smell.

Se você quiser, no próximo passo eu posso montar um **segundo notebook** já com a etapa de:
- gerar prompts automaticamente
- enviar para modelo
- parsear respostas
- calcular accuracy, precision, recall e F1


# Montagem dos Prompts

In [ ]:
display(
    amostra_blob_code[["sample_id", "target_smell", "target_label", "code"]].head(3)
)

,sample_id,target_smell,target_label,code
0,3908492,blob,1,"@Service (""knownRepositoryContentConsumer#create-archiva-metadata"")\n@Scope (""prototype"")\npublic class ArchivaMetad..."
1,5740146,blob,1,"@SuppressWarnings({""cast"", ""rawtypes"", ""serial"", ""unchecked"", ""unused""})\n@javax.annotation.Generated(value = ""Autog..."
2,6772589,blob,1,public static class APIRequestCreateMessengerProfile extends APIRequest<Page> {\n\n Page lastResponse = null;\n...


## Prompt 1

### Para classes:

In [18]:
def prompt_class_v1(code):
    return f"""
Analise a seguinte classe Java.

Pergunta: esta classe possui o code smell BLOB ou DATA CLASS?

Responda apenas neste formato:

Blob: SIM ou NÃO
Data Class: SIM ou NÃO

Código:
{code}
"""

### Para funções:

In [19]:
def prompt_function_v1(code):
    return f"""
Analise o seguinte método Java.

Pergunta: este método possui o code smell LONG METHOD ou FEATURE ENVY?

Responda apenas neste formato:

Long Method: SIM ou NÃO
Feature Envy: SIM ou NÃO

Código:
{code}
"""

## Prompt 2

### Para classes:

In [20]:
def prompt_class_v2(code):
    return f"""
Você é especialista em qualidade de código Java.

Definições:
- Blob: classe com muitas responsabilidades
- Data Class: classe com muitos atributos e pouca lógica

Analise a classe abaixo e responda:

Blob: SIM ou NÃO
Data Class: SIM ou NÃO

Código:
{code}
"""

### Para funções:

In [21]:
def prompt_function_v2(code):
    return f"""
Você é especialista em qualidade de código Java.

Definições:
- Long Method: método muito longo ou complexo
- Feature Envy: método que usa mais dados de outra classe do que da própria

Analise o método abaixo:

Long Method: SIM ou NÃO
Feature Envy: SIM ou NÃO

Código:
{code}
"""

## Prompt 3

### Para classes:

In [22]:
def prompt_class_v3(code):
    return f"""
Analise a classe Java abaixo.

Responda SOMENTE em JSON:

{{
  "blob": 0 ou 1,
  "data_class": 0 ou 1
}}

Código:
{code}
"""

### Para funções:

In [23]:
def prompt_function_v3(code):
    return f"""
Analise o método Java abaixo.

Responda SOMENTE em JSON:

{{
  "long_method": 0 ou 1,
  "feature_envy": 0 ou 1
}}

Código:
{code}
"""

## GERAR PROMPTS AUTOMATICAMENTE

In [24]:
def gerar_prompt(row, versao="v3"):
    code = row["code"]

    if row["type"] == "class":
        if versao == "v1":
            return prompt_class_v1(code)
        elif versao == "v2":
            return prompt_class_v2(code)
        else:
            return prompt_class_v3(code)

    else:
        if versao == "v1":
            return prompt_function_v1(code)
        elif versao == "v2":
            return prompt_function_v2(code)
        else:
            return prompt_function_v3(code)

In [ ]:
amostra_data_class_code["prompt"] = amostra_data_class_code.apply(
    lambda row: gerar_prompt(row, versao="v3"),
    axis=1
)

amostra_long_method_code["prompt"] = amostra_long_method_code.apply(
    lambda row: gerar_prompt(row, versao="v3"),
    axis=1
)

amostra_feature_envy_code["prompt"] = amostra_feature_envy_code.apply(
    lambda row: gerar_prompt(row, versao="v3"),
    axis=1
)

In [25]:
amostra_blob_code["prompt"] = amostra_blob_code.apply(
    lambda row: gerar_prompt(row, versao="v3"),
    axis=1
)

### Visualizar os prompts com os códigos

In [34]:
for i in range(3):
    print("\n--- PROMPT ---\n")
    print(amostra_blob_code.iloc[i]["prompt"])


--- PROMPT ---


Analise a classe Java abaixo.

Responda SOMENTE em JSON:

{
  "blob": 0 ou 1,
  "data_class": 0 ou 1
}

Código:
@Service ("knownRepositoryContentConsumer#create-archiva-metadata")
@Scope ("prototype")
public class ArchivaMetadataCreationConsumer
    extends AbstractMonitoredConsumer
    implements KnownRepositoryContentConsumer, RegistryListener
{
    private String id = "create-archiva-metadata";

    private String description = "Create basic metadata for Archiva to be able to reference the artifact";

    @Inject
    private ArchivaConfiguration configuration;

    @Inject
    private FileTypes filetypes;

    private Date whenGathered;

    private List<String> includes = new ArrayList<>( 0 );

    /**
     * FIXME: this could be multiple implementations and needs to be configured.
     */
    @Inject
    private RepositorySessionFactory repositorySessionFactory;

    /**
     * FIXME: this needs to be configurable based on storage type - and could also be instant

In [28]:
len(amostra_blob_code.iloc[0]["code"])

6666

### Juntar tudo em uma base única para copiar e colar

In [35]:
amostras_todas = pd.concat([
    amostra_blob_code,
    amostra_data_class_code,
    amostra_long_method_code,
    amostra_feature_envy_code
], ignore_index=True)

amostras_todas["expected"] = amostras_todas["target_label"].map({
    1: "SIM",
    0: "NÃO"
})

display(
    amostras_todas[[
        "sample_id",
        "type",
        "target_smell",
        "expected",
        "prompt"
    ]].head(10)
)

,sample_id,type,target_smell,expected,prompt
0,3908492,class,blob,SIM,"\nAnalise a classe Java abaixo.\n\nResponda SOMENTE em JSON:\n\n{\n ""blob"": 0 ou 1,\n ""data_class"": 0 ou 1\n}\n\nC..."
1,5740146,class,blob,SIM,"\nAnalise a classe Java abaixo.\n\nResponda SOMENTE em JSON:\n\n{\n ""blob"": 0 ou 1,\n ""data_class"": 0 ou 1\n}\n\nC..."
2,6772589,class,blob,SIM,"\nAnalise a classe Java abaixo.\n\nResponda SOMENTE em JSON:\n\n{\n ""blob"": 0 ou 1,\n ""data_class"": 0 ou 1\n}\n\nC..."
3,7356691,class,blob,SIM,"\nAnalise a classe Java abaixo.\n\nResponda SOMENTE em JSON:\n\n{\n ""blob"": 0 ou 1,\n ""data_class"": 0 ou 1\n}\n\nC..."
4,8119000,class,blob,SIM,"\nAnalise a classe Java abaixo.\n\nResponda SOMENTE em JSON:\n\n{\n ""blob"": 0 ou 1,\n ""data_class"": 0 ou 1\n}\n\nC..."
5,8198711,class,blob,SIM,"\nAnalise a classe Java abaixo.\n\nResponda SOMENTE em JSON:\n\n{\n ""blob"": 0 ou 1,\n ""data_class"": 0 ou 1\n}\n\nC..."
6,8408651,class,blob,SIM,"\nAnalise a classe Java abaixo.\n\nResponda SOMENTE em JSON:\n\n{\n ""blob"": 0 ou 1,\n ""data_class"": 0 ou 1\n}\n\nC..."
7,8436827,class,blob,SIM,"\nAnalise a classe Java abaixo.\n\nResponda SOMENTE em JSON:\n\n{\n ""blob"": 0 ou 1,\n ""data_class"": 0 ou 1\n}\n\nC..."
8,8962194,class,blob,SIM,"\nAnalise a classe Java abaixo.\n\nResponda SOMENTE em JSON:\n\n{\n ""blob"": 0 ou 1,\n ""data_class"": 0 ou 1\n}\n\nC..."
9,9199423,class,blob,SIM,"\nAnalise a classe Java abaixo.\n\nResponda SOMENTE em JSON:\n\n{\n ""blob"": 0 ou 1,\n ""data_class"": 0 ou 1\n}\n\nC..."


### 1. Criar 3 versões da base

In [36]:
amostras_v1 = amostras_todas.copy()
amostras_v1["prompt_version"] = "v1"
amostras_v1["prompt"] = amostras_v1.apply(
    lambda row: gerar_prompt(row, versao="v1"),
    axis=1
)

amostras_v2 = amostras_todas.copy()
amostras_v2["prompt_version"] = "v2"
amostras_v2["prompt"] = amostras_v2.apply(
    lambda row: gerar_prompt(row, versao="v2"),
    axis=1
)

amostras_v3 = amostras_todas.copy()
amostras_v3["prompt_version"] = "v3"
amostras_v3["prompt"] = amostras_v3.apply(
    lambda row: gerar_prompt(row, versao="v3"),
    axis=1
)

### 2. Juntar tudo

In [37]:
amostras_prompts = pd.concat([
    amostras_v1,
    amostras_v2,
    amostras_v3
], ignore_index=True)

### 3. Visualizar

In [40]:
display(amostras_prompts.head(2))

,sample_id,type,target_smell,target_label,group_label,code_name,repository,commit_hash,path,start_line,end_line,link,is_from_industry_relevant_project,blob,data class,feature envy,long method,code,download_error,expected,prompt,prompt_version
0,3908492,class,blob,1,com_smell,org.apache.archiva.consumers.metadata.ArchivaMetadataCreationConsumer,git@github.com:apache/archiva.git,d1242030bf232c0d9b68e4402188ee261924bf4b,/archiva-modules/archiva-base/archiva-consumers/archiva-metadata-consumer/src/main/java/org/apache/archiva/consumers...,59,264,https://github.com/apache/archiva/blob/d1242030bf232c0d9b68e4402188ee261924bf4b/archiva-modules/archiva-base/archiva...,1.0,1.0,0.0,NaN,NaN,"@Service (""knownRepositoryContentConsumer#create-archiva-metadata"")\n@Scope (""prototype"")\npublic class ArchivaMetad...",None,SIM,\nAnalise a seguinte classe Java.\n\nPergunta: esta classe possui o code smell BLOB ou DATA CLASS?\n\nResponda apena...,v1
1,5740146,class,blob,1,com_smell,org.apache.storm.generated.Assignment,git@github.com:apache/storm.git,dc56e32f3dcdd9396a827a85029d60ed97474786,/storm-client/src/jvm/org/apache/storm/generated/Assignment.java,26,1404,https://github.com/apache/storm/blob/dc56e32f3dcdd9396a827a85029d60ed97474786/storm-client/src/jvm/org/apache/storm/...,1.0,1.0,1.0,NaN,NaN,"@SuppressWarnings({""cast"", ""rawtypes"", ""serial"", ""unchecked"", ""unused""})\n@javax.annotation.Generated(value = ""Autog...",None,SIM,\nAnalise a seguinte classe Java.\n\nPergunta: esta classe possui o code smell BLOB ou DATA CLASS?\n\nResponda apena...,v1


#### Ver só colunas importantes

In [41]:
display(
    amostras_prompts[[
        "sample_id",
        "type",
        "target_smell",
        "prompt_version",
        "prompt"
    ]].head(10)
)

,sample_id,type,target_smell,prompt_version,prompt
0,3908492,class,blob,v1,\nAnalise a seguinte classe Java.\n\nPergunta: esta classe possui o code smell BLOB ou DATA CLASS?\n\nResponda apena...
1,5740146,class,blob,v1,\nAnalise a seguinte classe Java.\n\nPergunta: esta classe possui o code smell BLOB ou DATA CLASS?\n\nResponda apena...
2,6772589,class,blob,v1,\nAnalise a seguinte classe Java.\n\nPergunta: esta classe possui o code smell BLOB ou DATA CLASS?\n\nResponda apena...
3,7356691,class,blob,v1,\nAnalise a seguinte classe Java.\n\nPergunta: esta classe possui o code smell BLOB ou DATA CLASS?\n\nResponda apena...
4,8119000,class,blob,v1,\nAnalise a seguinte classe Java.\n\nPergunta: esta classe possui o code smell BLOB ou DATA CLASS?\n\nResponda apena...
5,8198711,class,blob,v1,\nAnalise a seguinte classe Java.\n\nPergunta: esta classe possui o code smell BLOB ou DATA CLASS?\n\nResponda apena...
6,8408651,class,blob,v1,\nAnalise a seguinte classe Java.\n\nPergunta: esta classe possui o code smell BLOB ou DATA CLASS?\n\nResponda apena...
7,8436827,class,blob,v1,\nAnalise a seguinte classe Java.\n\nPergunta: esta classe possui o code smell BLOB ou DATA CLASS?\n\nResponda apena...
8,8962194,class,blob,v1,\nAnalise a seguinte classe Java.\n\nPergunta: esta classe possui o code smell BLOB ou DATA CLASS?\n\nResponda apena...
9,9199423,class,blob,v1,\nAnalise a seguinte classe Java.\n\nPergunta: esta classe possui o code smell BLOB ou DATA CLASS?\n\nResponda apena...


#### Ver um prompt completo (sem corte)

In [ ]:
print(amostras_prompts.iloc[0]["prompt"])

#### Ver vários prompts

In [ ]:
for i in range(240):
    row = amostras_prompts.iloc[i]

    print("="*80)
    print(f"sample_id: {row['sample_id']}")
    print(f"smell: {row['target_smell']}")
    print(f"versão: {row['prompt_version']}")
    print(f"RESPOSTA ESPERADA: {row['expected']}")
    print("\nPROMPT:\n")
    print(row["prompt"])
    print("\n")

sample_id: 3908492
smell: blob
versão: v1
RESPOSTA ESPERADA: SIM

PROMPT:


Analise a seguinte classe Java.

Pergunta: esta classe possui o code smell BLOB ou DATA CLASS?

Responda apenas neste formato:

Blob: SIM ou NÃO
Data Class: SIM ou NÃO

Código:
@Service ("knownRepositoryContentConsumer#create-archiva-metadata")
@Scope ("prototype")
public class ArchivaMetadataCreationConsumer
    extends AbstractMonitoredConsumer
    implements KnownRepositoryContentConsumer, RegistryListener
{
    private String id = "create-archiva-metadata";

    private String description = "Create basic metadata for Archiva to be able to reference the artifact";

    @Inject
    private ArchivaConfiguration configuration;

    @Inject
    private FileTypes filetypes;

    private Date whenGathered;

    private List<String> includes = new ArrayList<>( 0 );

    /**
     * FIXME: this could be multiple implementations and needs to be configured.
     */
    @Inject
    private RepositorySessionFactory repos